# Neural--DTB nonlinear directed network game

This notebook studies an $N$-player game whose equilibria are not supplied in advance:

$$\Pi_i(x)=r_i x_i+\frac{\mu_i}{2}x_i^2-\frac14x_i^4+\beta_i x_i\tanh\!\left(\sum_jG_{ij}x_j\right).$$

The directed matrix $G$ has zero diagonal. The DTB drift is each player's own-action payoff gradient. Candidate equilibria are refined from terminal particles and classified by the spectrum of $Db$; this is not a complete enumeration of all equilibria.

## Block 1 -- Clone the Game-DTB branch and install requirements

In [ ]:
import pathlib, shutil, subprocess, sys

REPO_URL = "https://github.com/sun-mengwei/dtb-colab-experiments.git"
BRANCH = "codex/game-dynamics-dtb"
REPO_DIR = pathlib.Path("/content/dtb-colab-experiments")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
PROJECT_DIR = REPO_DIR / "dtb_game_dynamics_unnormalized"
%cd /content/dtb-colab-experiments/dtb_game_dynamics_unnormalized
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

## Block 2 -- Changeable controlled-experiment arguments

The baseline is 20 scalar players, a depth-4 MLP tangent generator, 512 particles, tangent-bundle size 128, and relative SVD tolerance $10^{-3}$. The interaction matrix is regenerated only when `NETWORK_SEED`, `DIM`, or network controls change.

In [ ]:
# Game setup
DIM = 20
NETWORK_DENSITY = 0.25
NETWORK_SCALE = 0.8
NETWORK_SEED = 17
NETWORK_BIAS_STD = 0.15
MU = 1.0
BETA = 0.8

# Particle dynamics
PARTICLES = 512
STEPS = 100
STEP_SIZE = 0.01
NOISE_STD = 0.05

# Initial NN architecture and tangent bundle
ARCHITECTURE = "mlp"
WIDTH = 32
DEPTH = 4
BASIS_SIZE = 128
SVD_RTOL = 1e-3

# Set to a positive divisor of STEPS to enable the source-matching reset
REFIT_INTERVAL = 0
CANDIDATE_SEEDS = 32
RUN_SDE_BASELINE = True
OUTPUT_DIR = "outputs/nonlinear_network_game"

## Block 3 -- Run Neural--DTB and the optional Euler--Maruyama baseline

In [ ]:
command = [
    sys.executable, "run_nonlinear_network_game.py",
    "--dim", str(DIM),
    "--network-density", str(NETWORK_DENSITY),
    "--network-scale", str(NETWORK_SCALE),
    "--network-seed", str(NETWORK_SEED),
    "--network-bias-std", str(NETWORK_BIAS_STD),
    "--network-mu", str(MU),
    "--network-beta", str(BETA),
    "--particles", str(PARTICLES),
    "--steps", str(STEPS),
    "--step-size", str(STEP_SIZE),
    "--noise-std", str(NOISE_STD),
    "--architecture", ARCHITECTURE,
    "--width", str(WIDTH),
    "--depth", str(DEPTH),
    "--basis-size", str(BASIS_SIZE),
    "--svd-rtol", str(SVD_RTOL),
    "--refit-interval", str(REFIT_INTERVAL),
    "--candidate-seeds", str(CANDIDATE_SEEDS),
    "--device", "auto",
    "--output-dir", OUTPUT_DIR,
]
if not RUN_SDE_BASELINE:
    command.append("--skip-sde-baseline")
subprocess.run(command, check=True)

## Block 4 -- Inspect distributions, DTB diagnostics, and equilibrium candidates

Red circles denote dynamically stable candidate roots; gold `X` markers denote unstable candidates. A candidate is dynamically stable when $\max\operatorname{Re}\lambda(Db)<0$. The CSV reports the local-Nash own-curvature test separately.

In [ ]:
import json, numpy as np, pandas as pd
from IPython.display import Image, display

for filename in [
    "dtb_snapshots.png",
    "sde_baseline_snapshots.png",
    "diagnostics.png",
    "network_equilibrium_analysis.png",
]:
    path = pathlib.Path(OUTPUT_DIR) / filename
    if path.exists():
        print(filename)
        display(Image(str(path)))

candidates = pd.read_csv(pathlib.Path(OUTPUT_DIR) / "equilibrium_candidates.csv")
display(candidates)
with open(pathlib.Path(OUTPUT_DIR) / "config.json") as handle:
    config = json.load(handle)
display(config["network_game"])

## Block 5 -- Read the saved numerical metrics

`history.npz` contains the exact interaction matrix and coefficients, all particle snapshots, projection residuals, retained SVD ranks, and optional baseline snapshots. An absent candidate does not imply that an equilibrium does not exist: forward particles preferentially expose attracting regions.

In [ ]:
history = np.load(pathlib.Path(OUTPUT_DIR) / "history.npz")
print("saved arrays:", sorted(history.files))
print("G shape:", history["network_interaction_matrix"].shape)
print("final projection residual:", history["projection_residuals"][-1])
print("final retained SVD rank:", history["retained_ranks"][-1])
print("candidate count:", len(candidates))